In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
# A program that takes a csv and trains models on it. Streamlined model selection.
# ==============================================================================

# STEFANOS: Disable unneeded modules
# #LazyPredict
# import lazypredict
# from lazypredict.Supervised import LazyRegressor
# from lazypredict.Supervised import LazyClassifier
# #Baysian Optimization
# from bayes_opt import BayesianOptimization
# Pandas stack
import os

# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
# import pandas_profiling

# STEFANOS: Disable unneeded modules
# #FastAI
# from fastai.tabular.all import *
# from fastai.tabular.core import *
# #Plots
# import matplotlib.pyplot as plt
# import seaborn as sns
# System
import os

# Fit an xgboost model
# STEFANOS: Disable unneeded modules
# from xgboost import XGBRegressor
# from xgboost import XGBClassifier
# from xgboost import plot_importance
# from sklearn.metrics import mean_squared_error
# from sklearn.metrics import roc_auc_score
# Random

# TabNet
# from fast_tabnet.core import *


In [ ]:
# Project Variables
# ===================================================================================================
PROJECT_NAME = "superstore"
VARIABLE_FILES = False
# Maximum amount of rows to take

## -- STEFANOS -- Remove sample

# SAMPLE_COUNT = 4000
FASTAI_LEARNING_RATE = 1e-1
AUTO_ADJUST_LEARNING_RATE = False
# Set to True automatically infer if variables are categorical or continuous
ENABLE_BREAKPOINT = True
# When trying to declare a column a continuous variable, if it fails, convert it to a categorical variable
CONVERT_TO_CAT = False
REGRESSOR = True
SEP_DOLLAR = False
SEP_COMMA = True
SHUFFLE_DATA = True

In [ ]:
%%time
### cell 0 ###

PARAM_DIR = "/home/dias-benchmarks/notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/input/superstore"
df = pd.read_csv(
    "/home/dias-benchmarks/notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/input/superstore/SampleSuperstore.csv"
)
factor = 10
df = pd.concat([df] * factor)
df.info()

In [ ]:
%%time
### cell 1 ###

if SEP_DOLLAR:
    # batch‐detect columns containing “$” in df_copy
    df_copy_str = df_copy.astype(str)
    mask = df_copy_str.apply(lambda s: s.str.contains(r"\$").any())
    dollar_cols = mask[mask].index.tolist()
    if dollar_cols:
        # remove $ and commas and convert to numeric in one vectorized step
        df_values = (
            df[dollar_cols]
            .astype(str)
            .replace(r"[\$,]", "", regex=True)
            .apply(pd.to_numeric, errors="coerce")
        )
        # rename and insert new columns in bulk
        df_values.columns = [col + "_no_dollar" for col in dollar_cols]
        df[df_values.columns] = df_values
    # preserve the original loop variable binding
    col = df_copy.columns[-1]

if SEP_COMMA:
    # batch‐detect columns containing “%” or “,” in df
    df_str = df.astype(str)
    mask = df_str.apply(lambda s: s.str.contains(r"[%\,]").any())
    proc_cols = mask[mask].index.tolist()
    if proc_cols:
        # remove % and commas and convert to numeric in one vectorized step
        df_values = (
            df[proc_cols]
            .astype(str)
            .replace(r"[%\,]", "", regex=True)
            .apply(pd.to_numeric, errors="coerce")
        )
        # rename and insert new columns in bulk
        df_values.columns = [col + "_processed" for col in proc_cols]
        df[df_values.columns] = df_values
    # preserve the original loop variable binding
    col = df.columns[-1]

In [ ]:
%%time
### cell 2 ###

df

In [ ]:
%%time
### cell 3 ###

df.isna().sum()

In [ ]:
%%time
### cell 4 ###

_ = df.select_dtypes(include=["number"]).corr()

In [ ]:
%%time
### cell 5 ###

df.head()

In [ ]:
%%time
### cell 6 ###

df.describe().T

In [ ]:
%%time
### cell 7 ###

df.columns

In [ ]:
%%time
### cell 8 ###

import os

# Create config files once
os.makedirs(PARAM_DIR, exist_ok=True)
for fname in ("cats.txt", "conts.txt", "cols_to_delete.txt"):
    open(os.path.join(PARAM_DIR, fname), "a").close()

# Initialize targets
target = ""
target_str = ""
targets = []

# Cache cols_to_delete file so we don’t reopen every iteration
cols_delete_path = os.path.join(PARAM_DIR, "cols_to_delete.txt")
with open(cols_delete_path) as f:
    cols_to_delete = f.read().splitlines()

# Iterate through potential target columns (continuous)
for col in reversed(df.columns[:-1]):
    # Try numeric conversion; skip if all NaN
    col_numeric = pd.to_numeric(df[col], errors="coerce")
    if not col_numeric.notna().any():
        continue
    df[col] = col_numeric
    target = col
    target_str = col.replace("/", "-")
    print(f"Target Variable: {target}")

    # Drop duplicates and optionally shuffle
    df.drop_duplicates(inplace=True)
    if SHUFFLE_DATA:
        df = df.sample(frac=1).reset_index(drop=True)

    # Cast boolean columns to uint8
    bool_cols = df.select_dtypes(include="bool").columns
    if bool_cols.size:
        df[bool_cols] = df[bool_cols].astype("uint8")

    # Drop unwanted columns
    df.drop(columns=cols_to_delete, errors="ignore", inplace=True)

    # Fill missing values
    df.fillna(0, inplace=True)

    # Auto-detect categorical vs continuous
    nunique = df.nunique()
    counts = df.count()
    ratio = nunique / counts
    cats = ratio.index[ratio < 0.05].tolist()
    conts = ratio.index[ratio >= 0.05].tolist()
    if target in cats:
        cats.remove(target)
    if target in conts:
        conts.remove(target)

    # Ensure target is numeric
    df[target] = pd.to_numeric(df[target], errors="coerce")

    # Override with external variable files if required
    if VARIABLE_FILES:
        with open(os.path.join(PARAM_DIR, "cats.txt")) as f:
            cats = f.read().splitlines()
        with open(os.path.join(PARAM_DIR, "conts.txt")) as f:
            conts = f.read().splitlines()

    # Vectorized conversion of continuous vars
    try:
        df[conts] = df[conts].apply(pd.to_numeric, errors="coerce")
    except Exception:
        for var in conts:
            try:
                df[var] = pd.to_numeric(df[var], errors="coerce")
            except Exception:
                print(f"Could not convert {var} to float.")

    # Experimental breakpoint logic
    if ENABLE_BREAKPOINT:
        cont_list = conts.copy()
        for var in conts:
            try:
                df[var] = pd.to_numeric(df[var], errors="coerce")
            except Exception:
                print(f"Could not convert {var} to float.")
                cont_list.remove(var)
                if CONVERT_TO_CAT:
                    cats.append(var)

In [ ]:
%%time
### cell 9 ###

df.isna().sum()